# Lecture 5 · Potential outcomes

Exercises 5.1 to 5.4 are conceptual, see [exercises.md](exercises.md). This notebook works through the potential outcome framework from the lecture instead.

In [ ]:
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA = Path("../../coding_tasks/Data/Out_Data")

---

## Example 5.7 · the table from the slides

Both potential outcomes for all seven students. In reality only one column per student is observable.

In [ ]:
slide = pd.DataFrame(
    {
        "D": [1, 0, 0, 0, 0, 1, 1],
        "Y0": [145, 100, 120, 110, 50, 120, 125],
        "Y1": [155, 100, 130, 100, 55, 130, 135],
    },
    index=[f"Student {i}" for i in range(1, 8)],
)
slide["ICE"] = slide["Y1"] - slide["Y0"]
slide["Y"] = np.where(slide["D"] == 1, slide["Y1"], slide["Y0"])
slide

In [ ]:
print(f"mean Y(0) = {slide['Y0'].mean():.2f}")
print(f"mean Y(1) = {slide['Y1'].mean():.2f}")
print(f"ATE       = {slide['ICE'].mean():.2f}")

Student 4 is hurt by the treatment. An average effect says nothing about an individual.

### The fundamental problem

What the researcher actually sees: one potential outcome per student, the other one missing.

In [ ]:
observed = slide[["D"]].copy()
observed["Y0"] = slide["Y0"].where(slide["D"] == 0)
observed["Y1"] = slide["Y1"].where(slide["D"] == 1)
observed["Y"] = slide["Y"]
observed

### The three estimands

In [ ]:
treated, control = slide[slide["D"] == 1], slide[slide["D"] == 0]

pd.Series(
    {
        "ATE": slide["ICE"].mean(),
        "ATT": treated["ICE"].mean(),
        "ATU": control["ICE"].mean(),
    }
).round(2)

### What the naive comparison gives

The difference in observed means is the only thing computable without the missing column.

In [ ]:
naive = treated["Y"].mean() - control["Y"].mean()
print(f"E[Y | D=1]  = {treated['Y'].mean():.2f}")
print(f"E[Y | D=0]  = {control['Y'].mean():.2f}")
print(f"naive       = {naive:.2f}")
print(f"true ATE    = {slide['ICE'].mean():.2f}")

Off by a factor of nine. The gap decomposes exactly into the effect on the treated plus how different the two groups would have been with no treatment at all.

    naive = ATT + ( E[Y(0) | D=1] - E[Y(0) | D=0] )

In [ ]:
att = treated["ICE"].mean()
selection_bias = treated["Y0"].mean() - control["Y0"].mean()

print(f"ATT            = {att:.2f}")
print(f"selection bias = {selection_bias:.2f}")
print(f"sum            = {att + selection_bias:.2f}")
print(f"naive          = {naive:.2f}")

The treated students would have scored 130 on average without exercising, the controls 95. Most of the measured 45 is that gap, not the treatment.

---

## The course data file

`lecture5_experimental_data.csv` has the same structure for eight students, but here the assignment was random.

In [ ]:
exp = pd.read_csv(DATA / "lecture5_experimental_data.csv").set_index("Student")
exp["ICE"] = exp["Y1"] - exp["Y0"]
exp

In [ ]:
t, c = exp[exp["D"] == 1], exp[exp["D"] == 0]

pd.Series(
    {
        "ATE": exp["ICE"].mean(),
        "ATT": t["ICE"].mean(),
        "naive": t["Y"].mean() - c["Y"].mean(),
        "selection bias": t["Y0"].mean() - c["Y0"].mean(),
    }
).round(2)

Selection bias is exactly zero, so the naive comparison lands on the true ATE. The two groups have identical mean Y(0) of 115, which is what a good assignment looks like.

Note this file also has a constant treatment effect of 10 for every student, so ATE, ATT and ATU all coincide. That is a property of the made-up data, not something randomization gives you.

---

## Beyond the lecture · the randomization distribution

With seven students and three treated there are only 35 possible assignments, so they can all be enumerated instead of simulated. For each one, compute what the naive comparison would have reported.

In [ ]:
def naive_over_all_assignments(y0, y1, n_treated):
    n = len(y0)
    out = []
    for idx in combinations(range(n), n_treated):
        d = np.zeros(n, dtype=int)
        d[list(idx)] = 1
        y = np.where(d == 1, y1, y0)
        out.append(y[d == 1].mean() - y[d == 0].mean())
    return np.array(out)


all_slide = naive_over_all_assignments(slide["Y0"].to_numpy(), slide["Y1"].to_numpy(), 3)

print(f"{len(all_slide)} possible assignments")
print(f"mean of the estimates = {all_slide.mean():.2f}  (true ATE {slide['ICE'].mean():.2f})")
print(f"standard deviation    = {all_slide.std():.2f}")
print(f"range                 = {all_slide.min():.2f} to {all_slide.max():.2f}")
print(f"the assignment on the slide gives {naive:.2f}, which is the largest of all {len(all_slide)}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(all_slide, bins=np.arange(-45, 50, 5), color="#4C78A8", edgecolor="white", linewidth=0.5)
ax.axvline(slide["ICE"].mean(), color="black", linewidth=1.5)
ax.axvline(naive, color="#B03A2E", linewidth=1.5, linestyle="--")
ax.set_xlabel("naive comparison (black: true ATE 5.0, red dashed: the slide's assignment 45.0)")
ax.set_yticks([])
ax.spines[["top", "right", "left"]].set_visible(False)
plt.tight_layout()
plt.show()

Two things to take from this.

- **Randomization is unbiased.** Averaged over all 35 assignments the naive comparison is exactly 5.0, the true ATE. Nothing about the data changed, only who was treated.
- **Unbiased is not the same as accurate.** The standard deviation across assignments is 24, and single draws range from -42.5 to 45. With seven students, one experiment says almost nothing.

The second point is the whole reason for lectures 6 and 7. Sample size turns an unbiased design into a useful one.

In [ ]:
all_exp = naive_over_all_assignments(exp["Y0"].to_numpy(), exp["Y1"].to_numpy(), 4)

pd.DataFrame(
    {
        "assignments": [len(all_slide), len(all_exp)],
        "mean estimate": [all_slide.mean(), all_exp.mean()],
        "true ATE": [slide["ICE"].mean(), exp["ICE"].mean()],
        "sd across assignments": [all_slide.std(), all_exp.std()],
    },
    index=["Example 5.7", "lecture5_experimental_data"],
).round(2)

Both are centred on their true ATE. The course data file is a bit tighter because its treatment effect is constant, so the only variation left comes from which students land in which group.